# FIT5230 M2 — Auxiliary Resolution-Alignment Experiments
**Dark.ModelBreakers · Wang Yunjie & Liu Junpeng · A100 Colab**

Our Theme 1 Dark primary plan directly improves StealthDiffusion LAO through a final-Control-VAE decoding objective and an additional CLIP proxy classification loss. Those changes are planned and are not implemented in this notebook. See the embedded English and Chinese M2 plans.

This notebook runs auxiliary RQ4 experiments: compare spectral alignment at a native image resolution with alignment at the detector input resolution. It adds a resize/blur EOT comparison and real JPEG evaluation.

**Current status:** software implemented and locally checked; this distributed notebook has no invented execution outputs. Run it on actual data/weights to produce results. M1 is completed and unchanged.

The four controlled methods operate in pixel space. A separate optional launcher runs the official StealthDiffusion baseline. This notebook does not implement EOT inside LAO or retrain Control-VAE.

**Before running:** select an A100 runtime, prepare official detector checkpoints and labelled real/generated image folders (or an existing split CSV) in Drive. Run development first, freeze settings, then select final test. An independent reserve is required for a public adaptive challenge.


## 1. Extract the included source bundle
All project code is embedded below, so uploading this notebook is sufficient. A SHA-256 check verifies its contents. Re-execution reuses identical files and refuses to replace locally modified code. Inspect `docs/M2_PLAN.md` for methodology and `docs/M2_UPSTREAM.md` for the upstream boundary.


In [ ]:
import base64, hashlib, io, json, os, subprocess, sys, zipfile
from pathlib import Path
BUNDLE_SHA256 = 'c3351449bb6414ac85181b00b092036793856f65e1c248acd22ead84ad4c73f9'
BUNDLE_BASE64 = '...'
blob = base64.b64decode(BUNDLE_BASE64)
assert hashlib.sha256(blob).hexdigest() == BUNDLE_SHA256
PROJECT = Path('/content') / ('FIT5230_M2_' + BUNDLE_SHA256[:10])
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    for name in archive.namelist():
        relative = Path(name)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Unsafe bundle member: ' + name)
        path = PROJECT / relative
        data = archive.read(name)
        if path.exists() and path.read_bytes() != data:
            raise RuntimeError('Existing modified file; preserve it before extracting: ' + str(path))
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists():
            path.write_bytes(data)
print('Project:', PROJECT, '| source files:', len(archive.namelist()))


## 2. Drive paths and experiment mode
Set `REAL_ROOT` to the photographs folder and `GENERATED_ROOT` to the generated-only folder (for GenImage these are commonly `nature` and `ai`). Existing `MANIFEST_PATH` is reused without changing its split. The new-manifest defaults select 200 real prototype images, 30 images per class for development, and 100 per class for test.

The current RQ4 run requires `resnet50.pth` and `efficientnet-b0.pth`. The optional full official baseline requires all six assets listed in the upstream guide. Weight files and datasets are not bundled.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/FIT5230_M2' #@param {type:"string"}
CHECKPOINT_PATH = '/content/drive/MyDrive/StealthDiffusion/checkpoints' #@param {type:"string"}
DRIVE_PROJECT = Path(DRIVE_PROJECT_PATH)
CHECKPOINT_DIR = Path(CHECKPOINT_PATH)
REAL_ROOT = '' #@param {type:"string"}
GENERATED_ROOT = '' #@param {type:"string"}
GENERATOR_SOURCE = 'SDv1.5' #@param {type:"string"}
EVALUATION_SPLIT = 'dev' #@param ["dev", "test"]
SEED = 5230 #@param {type:"integer"}
NATIVE_SIZE = 512 #@param [224, 512] {type:"raw"}
BATCH_SIZE = 4 #@param {type:"integer"}
EPSILON_PIXELS = 8 #@param [4, 8] {type:"raw"}
SPECTRAL_WEIGHT = 1.5 #@param {type:"number"}
MANIFEST_PATH = DRIVE_PROJECT / 'data' / 'manifest.csv'
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
print('Manifest:', MANIFEST_PATH, '| evaluation split:', EVALUATION_SPLIT)


## 3. Isolated Python/CUDA environment
Use the same interpreter for tests, detector loading and experiments. This avoids mixing legacy research dependencies with the notebook kernel. Package setup errors propagate. GPU availability and exact versions are checked after installation; A100 execution still needs this real runtime check.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
import shutil
UV = shutil.which('uv')
ENV_DIR = Path('/content/fit5230_m2_py310')
PYTHON = str(ENV_DIR / 'bin/python')
if not Path(PYTHON).exists():
    subprocess.run([UV, 'venv', '--python', '3.10', str(ENV_DIR)], check=True)
subprocess.run([UV, 'pip', 'install', '--python', PYTHON,
                'torch==2.2.2', 'torchvision==0.17.2', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([UV, 'pip', 'install', '--python', PYTHON,
                'numpy==1.26.4', 'Pillow==10.4.0', 'efficientnet-pytorch==0.7.1',
                'timm==0.6.12', 'natsort==8.4.0'], check=True)
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(PROJECT / 'm2')
ENV['PYTHONHASHSEED'] = str(SEED)
ENV['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
def execute(arguments):
    subprocess.run([PYTHON, *map(str, arguments)], cwd=PROJECT, env=ENV, check=True)
execute(['-c', "import torch; assert torch.cuda.is_available(), 'Select an A100 GPU runtime'; print('torch', torch.__version__, 'GPU', torch.cuda.get_device_name(0), 'GB', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))"])


## 4. Fetch the immutable official model definitions and run software checks
The official revision is fixed. This step downloads source only, not checkpoints or data. The M2 loader checks the official model definition hash and loads the supplied scalar heads strictly. Label polarity is then checked on the separate development images, not inferred from final-test results.


In [ ]:
UPSTREAM_COMMIT = 'a156fb0792732e1aabc5b94a0cbecf6829e79d73'
REFERENCE = PROJECT / 'third_party' / 'reference'
if not REFERENCE.exists():
    subprocess.run(['git', 'init', str(REFERENCE)], check=True)
    subprocess.run(['git', '-C', str(REFERENCE), 'remote', 'add', 'origin', 'https://github.com/wyczzy/StealthDiffusion.git'], check=True)
    subprocess.run(['git', '-C', str(REFERENCE), 'fetch', '--depth', '1', 'origin', UPSTREAM_COMMIT], check=True)
    subprocess.run(['git', '-C', str(REFERENCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
actual = subprocess.check_output(['git', '-C', str(REFERENCE), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == UPSTREAM_COMMIT
ENV['STEALTH_M2_TEST_UPSTREAM'] = str(REFERENCE)
execute(['-m', 'unittest', 'discover', '-s', 'm2/tests', '-v'])


## 5. Data split and configuration
Data splits are generated once and verified by decoded RGB content hashes. The loss prototype uses only real training images. Set the generator label accurately; multiple-generator studies should provide a manifest recording each image's source.

Native vs aligned losses use the same relative normalization to control size-related changes in loss scale. Its weight is provisional: select it on development, then keep it fixed for test. EOT samples identity, bilinear resize and blur; final evaluation uses true JPEG and bicubic resizing.


In [ ]:
if not MANIFEST_PATH.exists():
    if not REAL_ROOT or not GENERATED_ROOT:
        raise ValueError('Set REAL_ROOT and GENERATED_ROOT in cell 2, or supply an existing manifest.csv at MANIFEST_PATH.')
    execute(['m2/scripts/make_manifest.py', '--real-root', REAL_ROOT, '--generated-root', GENERATED_ROOT,
             '--output', MANIFEST_PATH, '--source', GENERATOR_SOURCE, '--seed', SEED])
for filename in ['resnet50.pth', 'efficientnet-b0.pth']:
    if not (CHECKPOINT_DIR / filename).is_file():
        raise FileNotFoundError('Missing required detector checkpoint: ' + str(CHECKPOINT_DIR / filename))
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
CONFIG = json.loads((PROJECT / 'm2/configs/rq4_a100.json').read_text())
RUN_DIR = DRIVE_PROJECT / 'runs' / f'rq4_{EVALUATION_SPLIT}_n{NATIVE_SIZE}_seed{SEED}_{stamp}'
CONFIG.update(manifest=str(MANIFEST_PATH), upstream_dir=str(REFERENCE), checkpoint_dir=str(CHECKPOINT_DIR),
              output_dir=str(RUN_DIR), evaluation_split=EVALUATION_SPLIT, seed=SEED,
              input_size=NATIVE_SIZE, batch_size=BATCH_SIZE)
CONFIG['attack'].update(epsilon=EPSILON_PIXELS/255, spectral_weight=SPECTRAL_WEIGHT)
CONFIG['source_bundle_sha256'] = BUNDLE_SHA256
CONFIG['upstream_commit'] = UPSTREAM_COMMIT
CONFIG_PATH = PROJECT / f'run_config_{stamp}.json'
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))
execute(['-m', 'stealth_m2.run', '--config', CONFIG_PATH, '--preflight-only'])


## 6. Run the controlled experiment
First use `dev`. Review the clean per-class performance for every detector, attack/quality tradeoffs and runtime, then freeze the configuration before setting `test`. A low surrogate development score stops the run rather than reversing labels automatically. Existing output directories cannot be overwritten. PNG rounding occurs before both saving and evaluation.


In [ ]:
execute(['-m', 'stealth_m2.run', '--config', CONFIG_PATH])
print('Results:', RUN_DIR)


## 7. Inspect measured results
Conditional ASR counts only generated images the same detector detected before attack under the same postprocessing condition. A zero eligible count produces `null`, not a successful attack. Unconditional escape and real-image FPR are separate. The run directory retains every per-image logit, quality value, duration and exact PNG.


In [ ]:
from IPython.display import display, Markdown, Image as DisplayImage
summary = json.loads((RUN_DIR / 'summary.json').read_text())
def rate(value):
    return 'n/a' if value is None else f'{100*value:.1f}%'
table = ['| Method | Detector | Condition | Success / eligible | ASR | Escape | Real FPR | SSIM |',
         '|---|---|---|---:|---:|---:|---:|---:|']
for row in summary:
    ssim = 'n/a' if row['mean_ssim'] is None else f"{row['mean_ssim']:.4f}"
    table.append(f"| {row['method']} | {row['detector']} | {row['condition']} | {row['success_count']}/{row['eligible_count']} | {rate(row['conditional_asr'])} | {rate(row['unconditional_escape_rate'])} | {rate(row['real_fpr'])} | {ssim} |")
display(Markdown('\n'.join(table)))
entries = json.loads((RUN_DIR / 'images.json').read_text())
sample_id = next(r['image_id'] for r in entries if r['label'] == 1)
for entry in entries:
    if entry['image_id'] == sample_id:
        print(entry['method'])
        display(DisplayImage(filename=str(RUN_DIR / entry['path']), width=256))


## 8. Native 224 control
At native224 with detector224, the two spectral branches should coincide. This control is needed to interpret RQ4 as an effect of our explicitly configured geometry. Upstream defaults to a 224→224 path; metadata declaring `sample_size=512` is not evidence that its default inference is mismatched.


In [ ]:
RUN_NATIVE224_CONTROL = True #@param {type:"boolean"}
if RUN_NATIVE224_CONTROL:
    control = json.loads(json.dumps(CONFIG))
    control.update(input_size=224, methods=['freq_native', 'freq_aligned'],
                   output_dir=str(RUN_DIR.with_name(RUN_DIR.name + '_control224')))
    control_path = CONFIG_PATH.with_name(CONFIG_PATH.stem + '_control224.json')
    control_path.write_text(json.dumps(control, indent=2))
    execute(['-m', 'stealth_m2.run', '--config', control_path])
    control_dir = Path(control['output_dir'])
    entries = json.loads((control_dir / 'images.json').read_text())
    index = {(r['image_id'], r['method']): control_dir / r['path'] for r in entries}
    ids = sorted({r['image_id'] for r in entries if r['method'] == 'freq_native'})
    equal = sum(index[(i, 'freq_native')].read_bytes() == index[(i, 'freq_aligned')].read_bytes() for i in ids)
    print(f'Native224 control: {equal}/{len(ids)} attack PNG pairs identical. Investigate any mismatch before claiming a resolution effect.')


## 9. Optional official full baseline
This runs the unchanged official LAO/Control-VAE algorithm with infrastructure patches. It is separate from the pixel-space experiment. It requires six checkpoints and a flat directory of AI-generated input images whose provenance/splits are recorded. It uses the upstream default 224px path. Its final perturbation is not constrained by our pixel-space 8/255 budget; report measured budgets separately.


In [ ]:
RUN_OFFICIAL_BASELINE = False #@param {type:"boolean"}
OFFICIAL_INPUTS = '' #@param {type:"string"}
SD_MODEL_ID = '' #@param {type:"string"}
if RUN_OFFICIAL_BASELINE:
    if not OFFICIAL_INPUTS:
        raise ValueError('Provide a flat AI-generated input folder with recorded provenance.')
    subprocess.run([UV, 'pip', 'install', '--python', PYTHON,
        'diffusers==0.12.1', 'huggingface_hub==0.23.4', 'transformers==4.30.2',
        'accelerate==0.20.3', 'torchattacks==3.5.1', 'lossers==0.0.9', 'eagerpy==0.30.0',
        'scipy==1.13.1', 'scikit-learn==1.5.1', 'opencv-python-headless==4.10.0.84',
        'tensorboardX==2.6.2.2', 'tqdm==4.66.5'], check=True)
    runtime = PROJECT / 'third_party' / 'official_runtime'
    if not runtime.exists():
        execute(['m2/scripts/prepare_upstream.py', 'prepare', '--source', REFERENCE,
                 '--destination', runtime, '--checkpoint-dir', CHECKPOINT_DIR])
    args = ['m2/scripts/prepare_upstream.py', 'baseline', '--runtime', runtime,
            '--images-dir', OFFICIAL_INPUTS, '--output-dir', DRIVE_PROJECT / 'runs' / ('official_' + stamp),
            '--surrogate', 'R', '--seed', SEED, '--iterations', '10', '--execute']
    if SD_MODEL_ID:
        args += ['--sd-model-id', SD_MODEL_ID]
    execute(args)
else:
    print('Official full pipeline not run. Current measured comparisons are pixel-space RQ4 methods.')


## 10. Static challenge and peer interaction
Build the public pack from a separate reserve run after selecting your method. It contains 20 real, 20 raw generated and 20 attacked PNGs, uniformly encoded at224; raw/attacked source IDs cannot overlap. The public zip contains no key. Static attacked escape is not the paired ASR used in the scientific experiment.

A Light team has not yet been selected. Use the guide in `docs/M2_PEER_ENGAGEMENT.md`; retain actual returned CSVs, discussion links and responses. No message is sent by this notebook.


In [ ]:
BUILD_PUBLIC_CHALLENGE = False #@param {type:"boolean"}
RESERVE_RUN_DIR = '' #@param {type:"string"}
if BUILD_PUBLIC_CHALLENGE:
    if not RESERVE_RUN_DIR:
        raise ValueError('Provide a separate reserve run; do not expose the final scientific test split.')
    challenge_dir = DRIVE_PROJECT / 'challenges' / ('reserve_' + stamp)
    execute(['-m', 'stealth_m2.challenge', 'build', '--run-dir', RESERVE_RUN_DIR,
             '--method', 'freq_aligned_eot', '--output', challenge_dir, '--per-group', '20', '--seed', SEED])
    print('Share only:', challenge_dir / 'challenge_public.zip')
    print('Keep private:', challenge_dir / 'private/key.csv')


## 11. Report and submission
Use only measured outputs from completed `run.json` records. Explain whether RQ4 is supported, unsupported or inconclusive; include failures and compute/quality costs. Keep pixel-space experiments and any official baseline clearly identified. JPEG was evaluation-only in the initial EOT variant.

Finish `docs/M2_ED_DRAFT.md` with actual results, demonstrations, checked Colab/resource links and peer interaction. Submit the resulting Ed URL as PDF/text to Moodle by **18 September2026,23:55MYT** according to the supplied assignment. This notebook does not publish or submit anything.

References: [StealthDiffusion](https://arxiv.org/abs/2408.05669), [EOT](https://arxiv.org/abs/1707.07397).
